In [1]:
import sys
!{sys.executable} -m pip install -U google-genai python-dotenv ipywidgets


[notice] A new release of pip is available: 25.0.1 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
load_dotenv() 
gemini_key = os.getenv("GEMINI_API_KEY")
print(gemini_key[:6])


AIzaSy


In [ ]:
# 인풋값 설정
import ipywidgets as widgets
from IPython.display import display, clear_output

LABEL_W = "80px"
FIELD_W = "700px"

common_style = {"description_width": LABEL_W}
text_layout = widgets.Layout(width=FIELD_W)
area_layout = widgets.Layout(width=FIELD_W, height="160px")

topic_w = widgets.Text(description="보고서 제목", layout=widgets.Layout(width="300px"))
purpose_w   = widgets.Text(description="보고서 목적", layout=widgets.Layout(width="600px"))
requirements_w = widgets.Textarea(description="요구사항", layout=widgets.Layout(width="600px", height="100px"), placeholder="예: 1. 2.")

btn = widgets.Button(description="확인", button_style="primary")
btn_box = widgets.HBox([btn], layout=widgets.Layout(justify_content="center", width="700px"))
out = widgets.Output()

result = {}  # 입력값 저장용

def on_click(_):
    result["topic"] = topic_w.value
    result["purpose"] = purpose_w.value
    result["requirements"] = requirements_w.value
    with out:
        clear_output()
        print("입력 완료")
        print(result)

btn.on_click(on_click)

display(topic_w, purpose_w, requirements_w, btn_box, out)


Text(value='', description='보고서 제목', layout=Layout(width='300px'))

Text(value='', description='보고서 목적', layout=Layout(width='600px'))

Textarea(value='', description='요구사항', layout=Layout(height='100px', width='600px'), placeholder='예: 1. 2.')

Output()

In [4]:
# 목차 생성

import os
from google import genai
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# 상태 저장
toc_state = {"toc": "", "final_toc": ""}

def get_inputs(default_topic="미정(보고서제목)", default_purpose="미정(보고서목적)", default_requirements="없음"):
    r = globals().get("result", {}) or {}

    def pick(key, widget_name):
        w = globals().get(widget_name, None)
        return (r.get(key) or (getattr(w, "value", "") if w else "") or "").strip()

    topic = pick("topic", "topic_w") or default_topic
    purpose = pick("purpose", "purpose_w") or default_purpose
    requirements = pick("requirements", "requirements_w") or default_requirements
    return topic, purpose, requirements

def make_prompt(mode, topic, purpose, requirements, toc=None, feedback=None):
    """mode: 'toc' | 'revise'"""
    if mode == "toc":
        return f"""
너는 컨설팅 보고서 작성 전문가다.
아래 입력을 바탕으로 '보고서 목차(TOC)'를 한국어로 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[출력 요구사항]
- 출력은 번호가 매겨진 목차 항목만 포함해야 한다.
- 제목, 설명, 안내문, 라벨을 절대 출력하지 말 것.
- "목차", "개정 목차", "개정목차"라는 문자열을 어떤 형태로도 출력하지 말 것.
- #, ##, ###, *, -, • 등 마크다운 기호를 절대 사용하지 말 것.
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 최소 5개 장(Chapter) 이상
- 장(1,2,3...)과 소절(1.1, 1.2...)로 구성
- 불필요한 설명문 없이 목차만 출력
- 도입부에 목차라는 표시 표현 생성 금지
- # 또는 * 절대 사용금지
""".strip()

    if mode == "revise":
        return f"""
너는 컨설팅 보고서 편집자다.
아래 기존 목차와 사용자 수정 지시를 반영하여 '개정 목차'를 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[기존 목차]
{toc}

[목차 수정]
{feedback}

[출력 요구사항]
- 출력은 번호가 매겨진 목차 항목만 포함해야 한다.
- 제목, 설명, 안내문, 라벨을 절대 출력하지 말 것.
- "목차", "개정 목차", "개정목차"라는 문자열을 어떤 형태로도 출력하지 말 것.
- #, ##, ###, *, -, • 등 마크다운 기호를 절대 사용하지 말 것.
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 불필요한 설명문 없이 목차만 출력
- 도입부에 개정목차라는 표시 표현 생성 금지
- # 또는 * 기호 절대 사용하지 말것
""".strip()

    raise ValueError("mode는 'toc' 또는 'revise'여야 합니다.")

def toc_ui(model_name="gemini-2.0-flash", width="800px"):
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY 환경변수가 없습니다. (.env 로드 또는 OS 환경변수 설정 필요)")

    topic, purpose, requirements = get_inputs()
    client = genai.Client(api_key=api_key)

    # 최초 목차 생성
    resp = client.models.generate_content(
        model=model_name,
        contents=make_prompt("toc", topic, purpose, requirements)
    )
    toc_state["toc"] = (resp.text or "").strip()
    toc_state["final_toc"] = toc_state["toc"]

    out = widgets.Output()

    def render():
        with out:
            clear_output()
            display(Markdown(" 생성된 목차\n\n```text\n" + toc_state["final_toc"] + "\n```"))

    feedback_w = widgets.Textarea(
        description="목차 수정",
        placeholder="추가, 수정, 삭제 가능합니다.",
        style={"description_width": "80px"},
        layout=widgets.Layout(width=width, height="120px")
    )

    apply_btn = widgets.Button(description="반영", button_style="primary")
    btn_box = widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center", width=width))

    def on_apply(_):
        feedback = (feedback_w.value or "").strip()
        if not feedback:
            toc_state["final_toc"] = toc_state["toc"]
        else:
            r = client.models.generate_content(
                model=model_name,
                contents=make_prompt("revise", topic, purpose, requirements, toc=toc_state["toc"], feedback=feedback)
            )
            toc_state["final_toc"] = (r.text or "").strip()

        # 최종 목차를 화면에 반영
        render()

    apply_btn.on_click(on_apply)

    render()
    display(out, feedback_w, btn_box)

# 실행
toc_ui(model_name="gemini-2.0-flash")


Output()

Textarea(value='', description='목차 수정', layout=Layout(height='120px', width='800px'), placeholder='추가, 수정, 삭제 …

In [5]:
print(toc_state["final_toc"])

1. 서론
1.1. 축구 전술의 중요성
1.2. 보고서의 목적 및 범위

2. 축구 전술의 기본 원리
2.1. 공격 전술의 기본 원리
2.2. 수비 전술의 기본 원리
2.3. 공수 전환의 중요성

3. 주요 축구 전술 종류
3.1. 4-4-2 포메이션
3.1.1. 4-4-2 포메이션의 장단점
3.1.2. 4-4-2 포메이션의 핵심 전략
3.2. 4-3-3 포메이션
3.2.1. 4-3-3 포메이션의 장단점
3.2.2. 4-3-3 포메이션의 핵심 전략
3.3. 3-5-2 포메이션
3.3.1. 3-5-2 포메이션의 장단점
3.3.2. 3-5-2 포메이션의 핵심 전략
3.4. 4-2-3-1 포메이션
3.4.1. 4-2-3-1 포메이션의 장단점
3.4.2. 4-2-3-1 포메이션의 핵심 전략

4. 현대 축구 전술의 동향
4.1. 점유율 축구의 발전
4.2. 압박 축구의 강화
4.3. 역습 축구의 진화

5. 전술 선택 시 고려사항
5.1. 선수단의 특성 분석
5.2. 상대 팀 분석
5.3. 경기 상황 고려

6. 결론
6.1. 효과적인 전술 운용을 위한 제언
6.2. 향후 축구 전술 발전 방향

7. 참고 문헌


In [7]:
# #5번 코드
import re
import time
from google import genai

topic, purpose, requirements = get_inputs()

def extract_chapters(final_toc: str):
    chapters = []
    pattern = re.compile(r"^\s*(\d+(?:\.\d+)*)\s*[\.\)]?\s+.+?:?\s*$")
    for line in (final_toc or "").splitlines():
        s = line.strip().replace("*", "")
        if pattern.match(s):
            chapters.append(s.rstrip(":").strip())
    return chapters



def make_prompt_for_report(chapter_title, topic, purpose, requirements, summary_context=""):
    return f"""
너는 대형 컨설팅펌 수준의 전문 보고서를 작성하는 최고 수준의 분석가이다.
아래 입력 정보를 바탕으로, 해당 소제목에 대해서
심층적이고 장문(긴 분량)의 고품질 보고서를 작성하라.

[입력]
- 해당 소제목: {chapter_title}
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 보고서 요구사항: {requirements}
- 이전 내용 요약: {summary_context}

[출력 요구사항]
1. REPORT
- “해당 소제목”에 정확히 대응하는 심층적이고 장문의 본문을 작성하라.
- 단순 설명을 지양하고, 원인·구조·배경·영향·시사점의 관점에서 심층적이고 전문적인 분석을 수행하라.
- 해당 주제에 대해 객관적 사실에 기반한 정확한 분석을 제공하라.
- 관련 데이터(통계, 조사 결과, 시장 동향)와 사례(기존 연구, 산업 사례, 국가·지역 비교)를 적극 활용하여 논리를 강화하라.
- 이전 내용 요약을 참고하여 논리적 흐름이 자연스럽게 이어지도록 작성하라.
- 검증된 사실 범위 내에서 가능한 한 많은 유의미한 정보를 포함하라.
- 내용은 보고서 목적에 직접적으로 부합하도록 구성하라.
- 장 제목과 소제목은 “1. 서론”, “1.1 정의”와 같은 형태로만 작성하며, ##, ### 등 마크다운 헤더 기호는 절대 사용하지 마라.
- 문자 * 및 강조를 위한 모든 기호는 절대 사용하지 마라.
- 모든 본문은 빈 줄 없이 작성하라.
- 문단은 필요한 경우에만 구분할 수 있으며, 문단이 바뀔 때에는 단일 줄바꿈만 허용한다.
- 문단 내부의 일반 서술 문장 사이에는 줄바꿈을 사용하지 마라.
- 나열이 필요한 경우에만 줄바꿈을 허용하며,
  각 항목은 반드시 -기호로 시작하고,
  항목 간에는 단일 줄바꿈만 사용하라.
- 연속된 줄바꿈(빈 줄)은 절대 사용하지 마라.
- 소제목 바로 아래에도 빈 줄을 두지 마라.

2. SUMMARY
- 해당 소제목의 핵심 내용을 정확히 2문장으로 요약하라.
- 분석 결과와 시사점이 포함되도록 하라.

3. SOURCES
- 해당 장의 본문, 하위 질문, 분석 및 답변을 작성하는 과정에서
  사실 확인, 수치 인용, 분석 틀 설정, 판단 근거로 실제 사용된 모든 외부 출처를
  MLA 형식으로 기재하라.
- 직접 인용 여부와 무관하게 분석에 활용되었으면 포함하라.
- 중복 출처는 1회만 기재하라.

[출력 형식]
반드시 아래 3개 블록만으로 출력하라. 블록의 순서, 대괄호 태그 표기, 시작/종료 태그는 그대로 유지하라.

[REPORT]
(해당 소제목 보고서 본문)
[/REPORT]

[SUMMARY]
(해당 소제목 핵심 요약 2문장)
[/SUMMARY]

[SOURCES]
(MLA 형식 출처 목록)
[/SOURCES]
""".strip()
def parse_response_blocks(text: str, chapter_title: str = ""):
    """
    우선순위
    1) 정상 케이스: [REPORT]..[/REPORT], [SUMMARY]..[/SUMMARY], [SOURCES]..[/SOURCES]
    2) 깨진 케이스(닫는 태그 누락 등):
       - [REPORT] ~ [SUMMARY] 직전까지 -> report
       - [SUMMARY] ~ [SOURCES] 직전까지 -> summary
       - [SOURCES] ~ 끝까지 -> sources
    3) 그것도 실패: "{chapter_title} 오류!"를 report로 반환
    """
    t = (text or "").strip()
    if not t:
        return f"{chapter_title} 오류!", "", ""

    def between_pair(t: str, a: str, b: str) -> str:
        if a not in t or b not in t:
            return ""
        return t.split(a, 1)[1].split(b, 1)[0].strip()

    # 1) 정상 케이스(닫는 태그가 모두 존재)
    report = between_pair(t, "[REPORT]", "[/REPORT]")
    summary = between_pair(t, "[SUMMARY]", "[/SUMMARY]")
    sources = between_pair(t, "[SOURCES]", "[/SOURCES]")

    # 정상 케이스는 report가 확보된 경우만 채택(요청사항에 맞춰 안정성 강화)
    if report:
        return report, summary, sources

    # 2) 깨진 케이스(시작 태그 기준 절단)
    if "[REPORT]" in t and "[SUMMARY]" in t and "[SOURCES]" in t:
        try:
            after_report = t.split("[REPORT]", 1)[1]
            report2 = after_report.split("[SUMMARY]", 1)[0].strip()

            after_summary = after_report.split("[SUMMARY]", 1)[1]
            summary2 = after_summary.split("[SOURCES]", 1)[0].strip()

            sources2 = after_summary.split("[SOURCES]", 1)[1].strip()

            if report2:
                return report2, summary2, sources2
        except Exception:
            pass

    # 3) 최종 실패
    return f"{chapter_title} 오류!", "", ""


def safe_extract_blocks(raw_text: str, chapter_title: str):
    # 기존 safe_extract_blocks 대신 parse_response_blocks의 fallback 규칙을 사용
    return parse_response_blocks(raw_text, chapter_title)


def generate_report_from_toc(
    topic: str,
    purpose: str,
    requirements: str,
    api_key: str,
    summary_context: str = "",
    model: str = "gemini-2.0-flash",
    dedup_sources: bool = False,
):
    client = genai.Client(api_key=api_key)
    chapters = extract_chapters(toc_state["final_toc"])

    report_parts = []
    summary_parts = []
    sources_parts = []

    updated_summary_context = summary_context or ""
    seen_sources = set()

    for chapter_title in chapters:
        prompt = make_prompt_for_report(
            chapter_title=chapter_title,
            topic=topic,
            purpose=purpose,
            requirements=requirements,
            summary_context=updated_summary_context,
        )

        response = client.models.generate_content(model=model, contents=prompt)
        time.sleep(2)  # 요청 간 간격

        raw_text = getattr(response, "text", "") or ""
        r, s, src = safe_extract_blocks(raw_text, chapter_title)

        if r:
            report_parts.append(r)

        # SUMMARY 누적(다음 호출 입력용)
        if s:
            summary_parts.append(s)
            updated_summary_context += f"- {chapter_title}: {s}\n"
        else:
            updated_summary_context += f"- {chapter_title}: (요약 없음)\n"

        # SOURCES 누적(출력용)
        if src:
            if dedup_sources:
                for line in src.splitlines():
                    item = line.strip()
                    if not item:
                        continue
                    if item not in seen_sources:
                        seen_sources.add(item)
                        sources_parts.append(item)
            else:
                sources_parts.append(src.strip())

    report = "\n\n".join(report_parts).strip() + ("\n\n" if report_parts else "")
    summary = "\n".join(summary_parts).strip()
    sources = "\n".join(sources_parts).strip() + ("\n" if sources_parts else "")

    return report, summary, sources, updated_summary_context

report, summary, sources, updated_summary_context = generate_report_from_toc(
    topic=topic,
    purpose=purpose,
    requirements=requirements,
    api_key=gemini_key,
    summary_context=updated_summary_context if 'updated_summary_context' in globals() else "",
    model="gemini-2.0-flash",
    dedup_sources=True,
)


# ===== 항목별 500자 프리뷰 출력 =====
print("=== report===")
print((report)[:500])

print("\n=== updated_summary_context ===")
print((updated_summary_context)[:500])

print("\n=== sources ===")
print((sources)[:500])




=== report===
1. 서론
축구는 단순한 스포츠를 넘어, 고도의 전략과 전술이 요구되는 복합적인 시스템이다. 팀의 승리를 위해서는 선수 개개인의 역량뿐만 아니라, 조직적인 움직임과 상대의 전략을 무력화할 수 있는 전술적 역량이 필수적이다. 따라서 축구 전술에 대한 이해는 선수, 코칭 스태프, 팬을 포함한 모든 축구 관계자에게 중요한 의미를 가진다. 본 보고서는 다양한 축구 전술의 종류를 심층적으로 분석하고, 각 전술의 특징, 장단점, 그리고 실제 경기에서의 적용 사례를 제시함으로써 축구 전술에 대한 이해를 돕고자 한다.

본 보고서는 다음과 같은 구조로 구성된다. 먼저, 축구 전술의 기본적인 개념과 중요성을 정의하고, 전술이 축구 경기 결과에 미치는 영향을 분석한다. 다음으로, 대표적인 축구 전술들을 포메이션, 공격 전술, 수비 전술, 전환 전술의 네 가지 범주로 나누어 설명하고, 각 전술의 핵심 원리, 선수 배치, 운용 방식, 그리고 성공 및 실패 사례를 상세하게 분석한다. 또한, 각 전술이 특정

=== updated_summary_context ===
- 1. 서론: 축구 전술은 팀의 승리를 위한 핵심 요소이며, 선수 개인의 역량과 더불어 조직적인 움직임과 전략적 사고를 통해 구현된다. 본 보고서는 축구 전술의 종류, 특징, 장단점, 적용 사례를 심층적으로 분석하여 축구 전술에 대한 이해를 높이고, 현대 축구 전술 트렌드 변화와 미래 발전 방향을 전망한다.
- 1.1. 축구 전술의 중요성: 축구 전술은 팀의 승리를 위한 핵심 요소로서, 선수 개인의 역량과 조직적인 움직임을 극대화하고 경기 흐름을 주도하는 데 중요한 역할을 한다. 효과적인 전술은 자원 배분의 효율성을 높이고, 팀워크를 강화하며, 상대 팀의 전략을 무력화하는 등 다양한 측면에서 팀의 경쟁력을 향상시킨다.
- 1.2. 보고서의 목적 및 범위: 본 보고서는 다양한 축구 전술의 종류, 특징, 장단점, 적용 사례를 심층적으로 분석하여 축구 전술에 대한 이해를 높이는 것을 목표로 한다. 각 

In [8]:
from docx import Document
import os
import re

def sanitize_filename(name: str, default="report"):
    name = (name or "").strip()
    if not name:
        return default
    # Windows 금지 문자 치환
    name = re.sub(r'[\\/:*?"<>|]+', "_", name)
    # 공백 정리
    name = re.sub(r"\s+", " ", name).strip()
    # 너무 긴 파일명 제한(선택)
    return name[:120]

def get_download_path(file_name="report.docx"):
    if os.name == "nt":  # Windows
        download_path = os.path.join(os.environ.get("USERPROFILE", ""), "Downloads")
    else:  # macOS, Linux
        download_path = os.path.join(os.environ.get("HOME", ""), "Downloads")

    if not download_path or not os.path.isdir(download_path):
        # Downloads가 없으면 현재 폴더로 fallback
        download_path = os.getcwd()

    os.makedirs(download_path, exist_ok=True)
    return os.path.join(download_path, file_name)

def add_multiline_text(doc: Document, text: str):
    for line in (text or "").splitlines():
        if line.strip() == "":
            doc.add_paragraph("")  # 빈 줄 유지
        else:
            doc.add_paragraph(line)

def save_report_and_sources(report_text: str, sources_text: str, topic: str, final_toc: str):
    file_name = f"{sanitize_filename(topic)}.docx"
    file_path = get_download_path(file_name)

    doc = Document()

    # 제목(문서 상단)
    doc.add_heading(topic, level=1)

    # 목차 텍스트(별도 섹션)
    doc.add_heading("TOC", level=1)
    add_multiline_text(doc, final_toc)

    # 본문
    doc.add_heading("Report", level=1)
    add_multiline_text(doc, report_text)

    # 출처
    doc.add_heading("Sources", level=1)
    add_multiline_text(doc, sources_text)

    doc.save(file_path)
    print(f"파일이 저장되었습니다: {file_path}")


topic, purpose, requirements = get_inputs()
final_toc = (toc_state.get("final_toc") or "").strip()

save_report_and_sources(report, sources, topic, final_toc)


파일이 저장되었습니다: C:\Users\mphk0\Downloads\축구 전술.docx
